# Retrieval-Augmented Generation (RAG) — Complete Assignment
**Student Name**: Dhruv Munjpara  
**Course**: Data Science & AI Master Program (TOPS Technologies)  
**ID**: 8849162891  

---

## Table of Contents
1. [Session 1: Introduction to RAG & Why It's Needed](#session-1)
2. [Session 2: Chunking & Embeddings](#session-2)
3. [Session 3: Vector Databases & Retrieval](#session-3)
4. [Session 4: Build PDF Chatbot](#session-4)


<a id='session-1'></a>
# Session 1: Introduction to RAG & Why It's Needed

### Task 1: What is RAG & Everyday App Example
**Definition**:
Retrieval-Augmented Generation (RAG) is an AI framework that connects Large Language Models (LLMs) to external knowledge sources (Vector DBs, APIs, documents). It retrieves relevant external document context before feeding it alongside the user query to the LLM to generate grounded, non-hallucinated answers.

**Zomato App Example**:
If a user asks: *"What is the best butter chicken near me open now with HDFC discount?"*, a standard LLM fails without live data. With RAG, Zomato retrieves live restaurant status, menu ratings, user location, and bank offers from its database, passing these facts into the LLM to generate an accurate recommendation.

--- 
### Task 2: Outdated LLM Limitations (IRCTC / IPL 2024)
3 Questions a 2022 LLM struggles to answer:
1. *"What is the departure schedule of the newly launched Ahmedabad to Mumbai Vande Bharat train?"*
2. *"Who won the IPL 2024 final match and who scored the highest runs?"*
3. *"What is the live seat availability status for today's Rajdhani Express?"*

**Problem**: Static LLM weights cannot learn post-cutoff events without expensive full retraining. RAG decouples knowledge retrieval from model parameters.

--- 
### Task 3: RAG Workflow Diagram
```
User Query ──> Retrieve Documents (Vector Search) ──> Combine Prompt & Context ──> LLM ──> Grounded Answer
```

--- 
### Task 4: Comparison Table
| Metric | RAG | Fine-tuning | Prompt Engineering |
|---|---|---|---|
| **Cost** | Low to Moderate | High (GPU Training) | Very Low |
| **Data Freshness** | Real-Time / Dynamic | Static / Delayed | Dynamic (Context Limit) |
| **Use Case Example** | Zomato Live Search / Notion AI Q&A | Domain Terminology Adaptation | Formatting / Tone Persona |
| **Speed** | Hours (Fast) | Days/Weeks (Slow) | Immediate |

--- 
### Task 5: Real-World Industry RAG Example
**Notion AI Q&A**: Notion indexes employee wiki pages into a vector database. When users ask questions, Notion performs vector search with permission filtering and passes retrieved chunks to Anthropic Claude / GPT-4 for hallucination-free answers with exact citations.

<a id='session-2'></a>
# Session 2: Chunking & Embeddings

In [ ]:
# Task 1: Python Chunking Function
def chunk_text(text: str, chunk_size: int, overlap: int) -> list[str]:
    words = text.split()
    if not words:
        return []
    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(words), step):
        chunk_words = words[i : i + chunk_size]
        chunks.append(" ".join(chunk_words))
        if i + chunk_size >= len(words):
            break
    return chunks

# Task 2: Chunk Privacy Policy (chunk_size=100, overlap=20)
with open('session_2_chunking_embeddings/privacy_policy_sample.txt', 'r', encoding='utf-8') as f:
    policy_text = f.read()

chunks = chunk_text(policy_text, chunk_size=100, overlap=20)
print(f"Total Chunks Created: {len(chunks)}")
for idx, c in enumerate(chunks[:2], 1):
    print(f"--- Chunk {idx} ---\n{c}\n")

In [ ]:
# Task 3 & 4: Embeddings & Cosine Similarity
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')
first_3_chunks = chunks[:3]
chunk_embeddings = model.encode(first_3_chunks)

print(f"Embedding Dimensions: {chunk_embeddings[0].shape}")
print(f"First 5 numbers of Chunk 1 Vector: {chunk_embeddings[0][:5]}")

query = "How is my personal data shared with third parties?"
query_emb = model.encode([query])
sims = cosine_similarity(query_emb, chunk_embeddings)[0]

for i, s in enumerate(sims, 1):
    print(f"Chunk {i} Similarity Score: {s:.4f}")
best_idx = np.argmax(sims)
print(f"\nMost Relevant: Chunk {best_idx + 1}")

### Task 5: Importance of Chunk Overlap
Chunk overlap ensures that sentences or facts split near chunk boundaries retain full semantic context. Without overlap, critical conditional statements (e.g., *"We do not sell data. However..."*) get cut in half, leading vector search to lose context and produce incorrect answers.

<a id='session-3'></a>
# Session 3: Vector Databases & Retrieval

In [ ]:
# Task 1: FAISS 4D Index
import faiss

vectors = np.array([[0.1, 0.8, 0.3, 0.5], [0.9, 0.2, 0.1, 0.4], [0.2, 0.1, 0.9, 0.7], [0.5, 0.5, 0.4, 0.6]], dtype=np.float32)
index = faiss.IndexFlatL2(4)
index.add(vectors)

query_v = np.array([[0.15, 0.75, 0.35, 0.48]], dtype=np.float32)
dist, idxs = index.search(query_v, 1)
print(f"FAISS Closest Vector Index: {idxs[0][0]}, Distance: {dist[0][0]:.4f}")

In [ ]:
# Task 2: ChromaDB Restaurant Search
import chromadb

client = chromadb.Client()
coll_rest = client.create_collection('zomato_restaurants_nb')
restaurants = [
    "Spicy Punjab Express: Famous for extremely hot and spicy chicken tikka and curry.",
    "Green Leaf Cafe: Fresh organic salads and cold-pressed juices.",
    "Ocean Catch: Butter garlic prawns and salmon.",
    "Sweet Treats Bakery: Chocolate cakes and pastries.",
    "Szechuan Dragon: Spicy Indo-Chinese noodles and chilli paneer."
]
coll_rest.add(documents=restaurants, ids=[f"r_{i}" for i in range(1, 6)])
res = coll_rest.query(query_texts=["spicy food"], n_results=1)
print("Top Restaurant for 'spicy food':", res['documents'][0][0])

In [ ]:
# Task 3: FAISS Top-2 Movies Search ('a scary space movie')
movies = [
    "Alien: Resurrection - Sci-fi horror about creatures hunting crew in space.",
    "Interstellar - Futuristic space travel near wormhole.",
    "The Hangover - Comedy bachelor party in Vegas.",
    "The Conjuring - Supernatural horror in haunted house.",
    "Gravity - Suspenseful space thriller about stranded astronaut."
]
movie_embs = model.encode(movies).astype(np.float32)
faiss.normalize_L2(movie_embs)
faiss_movies = faiss.IndexFlatIP(movie_embs.shape[1])
faiss_movies.add(movie_embs)

q_emb = model.encode(["a scary space movie"]).astype(np.float32)
faiss.normalize_L2(q_emb)
scores, m_idxs = faiss_movies.search(q_emb, 2)
print("Top 2 Retrieved Movies:")
for r in range(2):
    print(f" Rank {r+1}: {movies[m_idxs[0][r]]}")

In [ ]:
# Task 4: ChromaDB Instagram Captions ('healthy lifestyle')
coll_insta = client.create_collection('insta_captions_nb')
captions = [
    "Crushing my morning workout! Fueling with smoothie bowls and cardio. #fitness #healthylifestyle",
    "Wanderlust vibes! Exploring Bali beaches. #travel",
    "Double cheese smash burger with fries! 🍔",
    "Yoga session under the sun. Clean eating & balance! #wellness #healthyliving",
    "Vintage denim jacket for Sunday brunch! #fashion"
]
coll_insta.add(documents=captions, ids=[f"c_{i}" for i in range(1, 6)])
res_insta = coll_insta.query(query_texts=["healthy lifestyle"], n_results=1)
print("Top Instagram Caption:", res_insta['documents'][0][0])

<a id='session-4'></a>
# Session 4: Build PDF Chatbot

In [ ]:
# Task 1, 2, 3, 4: Complete PDF Q&A Pipeline
from PyPDF2 import PdfReader

# Task 1: Extract Text
reader = PdfReader('session_4_pdf_chatbot/sample_resume.pdf')
pdf_text = "\n".join([p.extract_text() for p in reader.pages if p.extract_text()])
print(f"Extracted {len(reader.pages)} pages, {len(pdf_text)} characters.")

# Task 2: Chunk by 500 characters
pdf_chunks = [pdf_text[i:i+500].strip() for i in range(0, len(pdf_text), 500) if pdf_text[i:i+500].strip()]
print(f"Total 500-char chunks: {len(pdf_chunks)}")

# Task 3: Embeddings (paraphrase-MiniLM-L6-v2)
model_para = SentenceTransformer('paraphrase-MiniLM-L6-v2')
pdf_embeddings = model_para.encode(pdf_chunks)

# Task 4: answer_question function
def answer_question(user_query: str):
    q_v = model_para.encode([user_query])
    sims = cosine_similarity(q_v, pdf_embeddings)[0]
    best_i = int(np.argmax(sims))
    print(f"Query: '{user_query}'")
    print(f"Best Similarity Score: {sims[best_i]:.4f}")
    print(f"Retrieved Chunk:\n{pdf_chunks[best_i]}")

answer_question("What projects has Dhruv worked on?")

### Task 5: Real-World Applications
1. **Legal Contracts**: Extracting specific indemnity & liability clauses from 100-page lease contracts.
2. **Medical Diagnostics**: Querying patient lab trend data across historical medical PDF reports.
3. **Textbook Q&A**: Explaining complex science principles directly from specific textbook chapters with page citations.